# Swan — Questionnaire, catalogue draft and test profiles

## MVP scope (Meet of 26/10/2026)

- **Concern:** acne-prone skin (adults, mild blemishes)
- **Output:** 3 face moisturizers per profile
- **Market:** France to start
- **Questionnaire:** see `questionnaire.md` (7 questions)

## Data sources

| Source | File | 
|---|---|
| Team moisturizer file (Open Beauty Facts audit 27/09 + CosIng matching) | `raw_data/moisturizers_with_cosing.csv` 
| CosIng reference (1,521 ingredients, retrieved 26/10) | `raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv`

## Settings to run my notebook

In [325]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [326]:
import pandas as pd

INPUT_PATH = "../raw_data/moisturizers_with_cosing.csv"
COSING_PATH = "../raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv"

SCOPE_COLUMN = "france"
NB_ROUTINE_PRODUCT = 3             # moisturizers shown per profile
BLEMISH_REFERRAL_LEVEL = 4         # Q3: from this level → referral (to validate)
MIN_INGREDIENTS = 5
MIN_PRODUCTS_PER_INGREDIENT = 5    # an ingredient becomes a column if at least 5 products contain it - just as a test, can be changed
AVOID_FRAGRANCE_INCLUDES_ALLERGENS = False  # Q7 'avoid fragrance' = only 'parfum' on the label
LIGHT_SKIN_TYPES = ["oily", "combination", "normal", "not_sure"]

# Word lists for the signals — names from EU Regulation 1223/2009, Annex III (fragrance allergens)
FRAGRANCE_ALLERGENS = ["linalool", "limonene", "citronellol", "geraniol", "citral", "coumarin", "eugenol",
                       "benzyl alcohol", "benzyl benzoate", "benzyl salicylate", "hexyl cinnamal", "farnesol",
                       "alpha-isomethyl ionone", "hydroxycitronellal", "cinnamal", "amyl cinnamal", "isoeugenol"]
ESSENTIAL_OILS = ["lavandula", "citrus", "mentha", "eucalyptus", "rosmarinus", "melaleuca", "pelargonium",
                  "cymbopogon", "cananga", "santalum", "pogostemon", "juniperus", "cinnamomum", "jasminum"]
ACNE_ACTIVES = ["salicylic acid", "niacinamide", "zinc pca", "zinc gluconate", "azelaic acid"]
# Texture: heavy ingredients = butters, waxes, oils, petrolatum (INCI order = amount above 1 %)
HEAVY_INGREDIENTS = ["petrolatum", "paraffinum liquidum", "paraffin", "mineral oil", "butter", "cera", "wax",
                     "lanolin", "squalane", "seed oil", "kernel oil", "fruit oil"]
RICH_NAME_WORDS = ["riche", "rich", "baume", "balm", "cold cream", "nourri"]
LIGHT_NAME_WORDS = ["gel", "fluide", "fluid", "matifiant", "light", "légère"]

## Load Data Set and check the columns

In [327]:
moist = pd.read_csv(INPUT_PATH, dtype={"code": str})
print(moist.shape)
print("France:", (moist["france"] == True).sum(), "| EU:", (moist["eu"] == True).sum())
moist[["code", "product_name", "brands", "ingredients_text"]].head()

(244, 17)
France: 114 | EU: 170


,code,product_name,brands,ingredients_text
0,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,Water. Glycerin. Copper Tripeptide-1. Phenoxye...
1,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,Water. Glycerin. Copper Tripeptide-1. Phenoxye...
2,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,"AQUA, NIACINAMIDE, DIBUTYL ADIPATE, GLYCERIN, ..."
3,8437018443420,RF Facial Cream,Masderm,"Aqua, Paraffinum Liquidum, Glyceryl Stearate, ..."
4,7891010255725,Hydro Boost,Neutrogena,"Aqua/água purificada, Dimethicone/dimeticona, ..."


## Check product type

In [328]:
def wrong_type(name):
    """Return the wrong product type found in the name ('' = looks like a face moisturizer)."""
    text = str(name).lower()
    if "nettoy" in text or "cleans" in text or "cleaning" in text or "foaming" in text:
        return "cleanser"
    elif "tonic" in text or "tonique" in text or "toner" in text:
        return "toner"
    elif "hand" in text or "mains" in text:
        return "hand"
    elif "yeux" in text or "eye" in text or "contour" in text:
        return "eye"
    elif text.startswith("huile") or " oil for" in text:
        return "oil"
    elif "mask" in text or "masque" in text or "mascarilla" in text:
        return "mask"
    elif "stick" in text or "gommage" in text or "désincrustant" in text:
        return "other type"
    else:
        return ""

moist["exclusion_reason"] = moist["product_name"].apply(wrong_type)
moist[moist["exclusion_reason"] != ""][["product_name", "exclusion_reason"]]

,product_name,exclusion_reason
0,PLU Laboratories GHK-Cu Face Tonic,toner
1,PLU Laboratories GHK-Cu Face Tonic,toner
34,Mains à Croquer,hand
103,Rice Bran Oil for Face,oil
125,Mascarilla hidratante nocturna,mask
127,Crème Nettoyante Désincrustante anti-points noirs,cleanser
144,Effaclar Deep Cleaning Foaming Crem,cleanser
146,huile de jojoba vierge,oil
147,huile de chanvre bio,oil
216,Soin anti-âge yeux rechargeur jeunesse et perf...,eye


In [329]:
#Check the exclusion based on product name for MVP
names = moist["product_name"].str.lower().fillna("")
moist["is_anti_ageing"] = names.str.contains("anti-âge|anti-age|rides|wrinkle|firming|lift")
moist["is_baby"] = names.str.contains("bébé|bebe|baby")
moist["is_face_and_body"] = names.str.contains("corps|body")
moist[["is_anti_ageing", "is_baby", "is_face_and_body"]].sum()

is_anti_ageing      36
is_baby              2
is_face_and_body     7
dtype: int64

In [330]:
#Check duplicates: same name + brand = possible duplicate, NOT excluded
# (different codes can be different sizes or formulas; Lina checks them by hand)
moist["name_brand"] = names + " | " + moist["brands"].str.lower().fillna("")
moist["possible_duplicate"] = moist["name_brand"].duplicated(keep=False)
print("Products with a possible duplicate:", moist["possible_duplicate"].sum())

Products with a possible duplicate: 14


In [331]:
#Products whose name doesn't say what they are (e.g. "Sébium sensitive") → assumption by looking at the name
product_words = "crème|creme|cream|crema|krem|moistur|hydrat|gel|fluid|baume|balm|soin|lotion|emulsion"
moist["to_check_by_hand"] = ~names.str.contains(product_words)
print("To check by hand:", moist["to_check_by_hand"].sum())
moist[moist["to_check_by_hand"]][["code", "product_name", "brands"]]

To check by hand: 62


,code,product_name,brands
0,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories
1,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories
4,7891010255725,Hydro Boost,Neutrogena
8,5201143734776,ΕΝΥΔΑΤΙΚΗ ΚΡΕΜΑ ΣΕ ΜΟΡΦΗ ΤΖΕΛ,Diadermine
29,3614272400467,Life plankton elixir,Biotherm
...,...,...,...
230,3600541295742,Rénovateur Anti-Taches Nuit,Garnier
235,3178041307116,Lift+ Ultra Repulpant Nuit,Diadermine
237,3040690661546,Stick douceur visage junior,Doctissimo parapharmacie
238,3574660219623,Visibly Clear Désincrustant Points Noirs,Neutrogena


## One Name per ingredient

In [332]:
cosing = pd.read_csv(COSING_PATH, dtype=str)
cosing["inci"] = cosing["inci_name"].str.lower()
cosing = cosing.drop_duplicates("inci")      # 3 names appear twice in the reference
inci_names = list(cosing["inci"])
print(len(inci_names), "INCI names in the reference")

1518 INCI names in the reference


In [333]:
#Split the ingredient into a list
def split_ingredients(text):
    """Turn the ingredient text into a list of names."""
    text = str(text).lower()
    text = text.replace("ingredients:", "").replace("ingrédients:", "")
    text = text.replace("•", ",")                                # some labels use • instead of ,
    text = text.replace("1,2-hexanediol", "1-2-hexanediol")     # this comma is part of the name
    if "," not in text:
        text = text.replace(". ", ",")                          # some labels use '. ' instead of ','
    names = []
    for piece in text.split(","):
        name = piece.strip().strip(".").strip()
        if name != "":
            names.append(name.replace("1-2-hexanediol", "1,2-hexanediol"))
    return names

ALIASES = {
    "water": "aqua", "eau": "aqua",
    "fragrance": "parfum", "perfume": "parfum",
    "glycerine": "glycerin",
    "shea butter": "butyrospermum parkii butter", "butyrospermum parkii": "butyrospermum parkii butter",
    "alcohol denat": "alcohol denat.",
}

def to_inci(name):
    """Clean one ingredient name and return its INCI name when we know it."""
    name = name.lower().replace("*", "").strip().strip(".").strip()
    if "(" in name and ")" in name:                      # 'parkii (shea) butter' -> 'parkii butter'
        name = name.split("(")[0] + name.split(")")[-1]
    elif "(" in name:                                    # 'aqua (water' (bracket not closed) -> 'aqua'
        name = name.split("(")[0]
    name = " ".join(name.split())
    name = ALIASES.get(name, name)
    if name not in inci_names and "/" in name:           # 'aqua/water/eau' -> 'aqua'
        first = ALIASES.get(name.split("/")[0].strip(), name.split("/")[0].strip())
        if first in inci_names:
            name = first
    return name

print(split_ingredients("Water. Glycerin. Phenoxyethanol."))
print(to_inci("Aqua / Water"), "|", to_inci("butyrospermum parkii (shea) butter*"), "|", to_inci("parfum (fragrance)."))

['water', 'glycerin', 'phenoxyethanol']
aqua | butyrospermum parkii butter | parfum


## Generate the table with one raw - one product x one ingredient 

In [334]:
rows = []
for code, text in zip(moist["code"], moist["ingredients_text"]):
    position = 1
    for raw_name in split_ingredients(text):
        rows.append({"code": code, "position": position, "raw_name": raw_name, "inci": to_inci(raw_name)})
        position = position + 1

long = pd.DataFrame(rows)
long = long[long["inci"] != ""]
long["in_cosing"] = long["inci"].isin(inci_names)

print("Rows:", len(long))
print("Different names before / after:", long["raw_name"].nunique(), "/", long["inci"].nunique())
print("Found in CosIng:", round(long["in_cosing"].mean() * 100, 1), "%   (team matching: 84.8 %)")

Rows: 6472
Different names before / after: 1747 / 1505
Found in CosIng: 91.6 %   (team matching: 84.8 %)


In [335]:
long[~long["in_cosing"]]["inci"].value_counts().head(20)

inci
palmitoyl oligopeptide                            5
_prunus amygdalus_ dulcis oil                     3
brassica juncea seed oil                          2
lysine hci                                        2
coco caprylate                                    2
ingredients/ aqua                                 2
ginkgo bilboa leaf extract                        2
caprylic                                          2
cl 16035                                          2
aus kontrolliert biologischem anbau               2
caprylic triglyceride                             2
capric triglyceride                               2
helianthus annus seed oil                         2
acrylates / c10-30 alkyl acrylate crosspolymer    2
avene thermal spring water                        2
glycine soja                                      2
cl 77891                                          2
alpha-isomethyl lonone                            2
butylphenyl methylproponal                        2
lonicer

In [336]:
moist["nb_ingredients"] = moist["code"].map(long.groupby("code")["inci"].count()).fillna(0)
moist["coverage"] = moist["code"].map(long.groupby("code")["in_cosing"].mean()).fillna(0)
moist.loc[(moist["exclusion_reason"] == "") & (moist["nb_ingredients"] < MIN_INGREDIENTS), "exclusion_reason"] = "list too short"

moist["in_scope"] = (moist["exclusion_reason"] == "") & (moist[SCOPE_COLUMN] == True)
print(moist["exclusion_reason"].value_counts())
print("In scope (" + SCOPE_COLUMN + "):", moist["in_scope"].sum(), "| same rules in the EU:", ((moist["exclusion_reason"] == "") & (moist["eu"] == True)).sum())

exclusion_reason
                  219
list too short     11
oil                 4
cleanser            3
toner               2
other type          2
hand                1
mask                1
eye                 1
Name: count, dtype: int64
In scope (france): 99 | same rules in the EU: 152


## TRY to standardize based on the questionnaire 

In [337]:
moist["inci_text"] = moist["code"].map(long.groupby("code")["inci"].apply(" | ".join)).fillna("")
moist["top5_text"] = moist["code"].map(long[long["position"] <= 5].groupby("code")["inci"].apply(" | ".join)).fillna("")

moist["has_parfum"] = moist["inci_text"].str.contains("parfum")
moist["has_fragrance_allergen"] = moist["inci_text"].str.contains("|".join(FRAGRANCE_ALLERGENS))
moist["has_essential_oil"] = moist["inci_text"].str.contains("|".join(ESSENTIAL_OILS))
moist["has_acne_active"] = moist["inci_text"].str.contains("|".join(ACNE_ACTIVES))

#review the texture details
def count_heavy(ingredients):
    """How many ingredients of the list are heavy and can identify a rich texture?"""
    count = 0
    for ingredient in ingredients:
        for heavy in HEAVY_INGREDIENTS:
            if heavy in ingredient:
                count = count +1
                break
    return count


def find_texture(inci_text, name):
    """Identify the texture rich, light or medium from the product name and the first ingredients"""

    name = str(name).lower()
    for word in RICH_NAME_WORDS:
        if word in name:
            return "rich"
    for word in LIGHT_NAME_WORDS:
        if word in name and "baume" not in name and "gelée" not in name:
            return "light"

    #if not look at the first ingredients:
    ingredients = inci_text.split(" | ")
    heavy_top3 = count_heavy(ingredients[:3])
    heavy_top5 = count_heavy(ingredients[:5])

    if ingredients[0] != "aqua" or heavy_top3 >= 1 or heavy_top5 >= 2:
        return "rich"
    elif heavy_top5 == 0:
        return "light"
    else:
        return "medium"

textures = [find_texture(text, name) for text, name in zip(moist["inci_text"], moist["product_name"])]
moist["is_rich"] = [texture == "rich" for texture in textures]
moist["is_light"] = [texture == "light" for texture in textures]

signals = ["has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_rich", "is_light",
           "is_anti_ageing", "is_baby", "is_face_and_body"]
(moist[moist["in_scope"]][signals].mean() * 100).round(1)

has_parfum                79.8
has_fragrance_allergen    52.5
has_essential_oil         10.1
has_acne_active           11.1
is_rich                   46.5
is_light                  41.4
is_anti_ageing            24.2
is_baby                    2.0
is_face_and_body           6.1
dtype: float64

## Put ingredients as a column 

In [338]:
count = long.groupby("inci")["code"].nunique()
frequent = list(count[count >= MIN_PRODUCTS_PER_INGREDIENT].index)
print("Different ingredients:", len(count), "→ columns kept:", len(frequent))

frequent_rows = long[long["inci"].isin(frequent)]
matrix = pd.crosstab(frequent_rows["code"], frequent_rows["inci"])
matrix = (matrix > 0).astype(int)          # 1 = the product contains it, 0 = it doesn't
print(matrix.shape)

Different ingredients: 1505 → columns kept: 252
(235, 252)


In [339]:
#Products whose name doesn't say what they are (e.g. "Sébium sensitive") → for Lina to check by hand
names = moist["product_name"].str.lower().fillna("")
product_words = "crème|creme|cream|crema|krem|moistur|hydrat|gel|fluid|baume|balm|soin|lotion|emulsion"
moist["to_check_by_hand"] = ~names.str.contains(product_words)
print("To check by hand:", moist["to_check_by_hand"].sum())

To check by hand: 62


In [340]:
columns = ["code", "product_name", "brands", "france", "eu", "in_scope", "exclusion_reason", "possible_duplicate",
           "to_check_by_hand", "nb_ingredients", "coverage", "inci_text"] + signals
moist[columns].to_csv("../raw_data/draft_for_lina_moisturizers.csv", index=False)
long.to_csv("../raw_data/draft_for_lina_ingredients_long.csv", index=False)
matrix.to_csv("../raw_data/draft_for_lina_ingredient_matrix.csv")
print("3 draft files saved")

3 draft files saved


## The questionnaire 

In [341]:
QUESTIONNAIRE = {
    "is_adult": [True, False],
    "concern": ["acne_prone", "other"],
    "blemish_level": [1, 2, 3, 4, 5],
    "referral_flags": [[], ["prescribed_treatment"], ["unusual_lesion"], ["pregnant_or_breastfeeding"]],
    "skin_type": ["oily", "combination", "normal", "dry", "not_sure"],
    "reactivity": [True, False],
    "avoid": [[], ["fragrance"], ["essential_oils"], ["fragrance", "essential_oils"]],
}

def check_profile(profile):
    """List the problems of a profile (empty list = valid)."""
    problems = []
    for field in QUESTIONNAIRE:
        if field not in profile:
            problems.append("missing " + field)
        elif type(profile[field]) == list:                  # Q4 and Q7: several answers possible
            allowed = []
            for option in QUESTIONNAIRE[field]:
                allowed = allowed + option
            for item in profile[field]:
                if item not in allowed:
                    problems.append("wrong value for " + field + ": " + item)
        elif profile[field] not in QUESTIONNAIRE[field]:
            problems.append("wrong value for " + field)
    return problems

print(check_profile({"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                     "skin_type": "oily", "reactivity": True, "avoid": []}))
print(check_profile({"is_adult": True, "concern": "acne", "skin_type": "very oily"}))

[]
['wrong value for concern', 'missing blemish_level', 'missing referral_flags', 'wrong value for skin_type', 'missing reactivity', 'missing avoid']


**Data contract** — what each answer needs from the catalogue (for Lina, Loubna, Imane):

| Question | Profile field | Catalogue column(s) |
|---|---|---|
| Q1 age · Q2 concern · Q3 blemishes · Q4 special cases | `is_adult`, `concern`, `blemish_level`, `referral_flags` | none (referral rule) |
| Q5 skin type | `skin_type` | `is_rich`, `is_light` |
| Q6 reactivity | `reactivity` | `has_parfum`, `has_essential_oil` |
| Q7 avoid | `avoid` | `has_parfum`, `has_fragrance_allergen`, `has_essential_oil` |
| Concern (acne-prone) | — | `has_acne_active`, `is_anti_ageing` |

In [342]:
contract = ["is_rich", "is_light", "has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_anti_ageing"]
print("Columns missing in the catalogue:", [column for column in contract if column not in moist.columns])

Columns missing in the catalogue: []


In [343]:
def profile_outcome(profile):
    """'out of scope', 'referral' or 'eligible', in the order of the questionnaire."""
    if profile["is_adult"] == False:
        return "referral"                  # Meet 26/10: under 18 = referral
    elif profile["concern"] != "acne_prone":
        return "out of scope"              # not a medical referral: Swan only covers acne-prone skin
    elif profile["blemish_level"] >= BLEMISH_REFERRAL_LEVEL:
        return "referral"
    elif len(profile["referral_flags"]) > 0:
        return "referral"
    else:
        return "eligible"

def eligible_products(profile, catalogue):
    """In-scope products left after the 'avoid' answers."""
    selected = catalogue[catalogue["in_scope"]]
    avoid = profile.get("avoid", [])        # Q7 is optional: default = nothing to avoid
    if "fragrance" in avoid:
        selected = selected[~selected["has_parfum"]]
        if AVOID_FRAGRANCE_INCLUDES_ALLERGENS:
            selected = selected[~selected["has_fragrance_allergen"]]
    if "essential_oils" in avoid:
        selected = selected[~selected["has_essential_oil"]]
    return selected

def simple_score(profile, selected):
    """PLACEHOLDER score — Loubna's score will replace it (T15). Each point has a reason."""
    score = selected["has_acne_active"].astype(int) - selected["is_anti_ageing"].astype(int)
    if profile["reactivity"] == True:                     # fragrance ranked lower, not removed (removal = Q7)
        score = score - selected["has_parfum"].astype(int) - selected["has_essential_oil"].astype(int)
    if profile["skin_type"] in LIGHT_SKIN_TYPES:          # light preferred, rich ranked lower (not removed)
        score = score + selected["is_light"].astype(int) - selected["is_rich"].astype(int)
    elif profile["skin_type"] == "dry":
        score = score + selected["is_rich"].astype(int)
    return score

def recommend(profile, catalogue):
    """Answers -> 'out of scope', 'referral', 'not enough products', or the top moisturizers."""
    outcome = profile_outcome(profile)
    if outcome != "eligible":
        return outcome
    selected = eligible_products(profile, catalogue).copy()
    selected["score"] = simple_score(profile, selected)
    selected = selected.sort_values(["score", "coverage"], ascending=False)
    selected = selected.drop_duplicates("name_brand")      # the same product never twice in a top 3
    if len(selected) < NB_ROUTINE_PRODUCT:
        return "not enough products"
    return selected.head(NB_ROUTINE_PRODUCT)

## Test some testing profiles


Avoid fragrance' (Q7): removes products with `parfum` only (19 products left in France)
Reactive skin (Q6 = yes): fragrance and essential oils ranked lower, not removed
Oily, combination, normal, not sure:  light texture preferred, rich ranked lower (not removed: only 5 products are tagged light)
Dry: rich texture preferred


In [344]:
# Recommendation profiles: every skin type x reactivity x (nothing / fragrance) to avoid,
profiles = []
number = 1
for skin_type in QUESTIONNAIRE["skin_type"]:
    for reactivity in [True, False]:
        for avoid in [[], ["fragrance"]]:
            profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "recommendation",
                             "is_adult": True, "concern": "acne_prone", "blemish_level": number % 3 + 1,
                             "referral_flags": [], "skin_type": skin_type, "reactivity": reactivity,
                             "avoid": avoid, "expected": "products"})
            number = number + 1

# The 2 other 'avoid' answers, once each
for avoid in [["essential_oils"], ["fragrance", "essential_oils"]]:
    profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "recommendation",
                     "is_adult": True, "concern": "acne_prone", "blemish_level": 2,
                     "referral_flags": [], "skin_type": "combination", "reactivity": True,
                     "avoid": avoid, "expected": "products"})
    number = number + 1

# Edge cases: one per referral rule, + 2 reasons at once
edge_cases = [
    ("under 18",                       {"is_adult": False}),
    ("other concern",                  {"concern": "other", "expected": "out of scope"}),
    ("blemishes = 4",                  {"blemish_level": 4}),
    ("blemishes = 5",                  {"blemish_level": 5}),
    ("prescribed treatment",           {"referral_flags": ["prescribed_treatment"]}),
    ("unusual lesion",                 {"referral_flags": ["unusual_lesion"]}),
    ("pregnant",                       {"referral_flags": ["pregnant_or_breastfeeding"]}),
    ("pregnant + prescribed treatment", {"referral_flags": ["pregnant_or_breastfeeding", "prescribed_treatment"]}),
]
for name, change in edge_cases:
    profile = {"profile_id": "P" + str(number).zfill(2), "kind": "edge case: " + name,
               "is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
               "skin_type": "oily", "reactivity": False, "avoid": [], "expected": "referral"}
    profile.update(change)
    profiles.append(profile)
    number = number + 1

profiles.append({"profile_id": "P" + str(number).zfill(2), "kind": "edge case: blemishes = 3 (just below)",
                 "is_adult": True, "concern": "acne_prone", "blemish_level": 3, "referral_flags": [],
                 "skin_type": "combination", "reactivity": True, "avoid": [], "expected": "products"})

def expected_traits(profile):
    """What a good product should respect for this profile, from the decisions above."""
    if profile["expected"] != "products":
        return "no products"
    traits = []
    if "fragrance" in profile["avoid"]:
        traits.append("no parfum")
    if "essential_oils" in profile["avoid"]:
        traits.append("no essential oil")
    if profile["reactivity"] == True:
        traits.append("fragrance-free first")
    if profile["skin_type"] in LIGHT_SKIN_TYPES:
        traits.append("not rich")
    elif profile["skin_type"] == "dry":
        traits.append("rich")
    return ", ".join(traits)

for profile in profiles:
    profile["expected_traits"] = expected_traits(profile)

profiles_table = pd.DataFrame(profiles)
print("Profiles:", len(profiles_table))
print(profiles_table["expected"].value_counts())
print(pd.crosstab(profiles_table["reactivity"], profiles_table["avoid"].astype(str)))   # check: reactivity and avoid are crossed
profiles_table

Profiles: 31
expected
products        23
referral         7
out of scope     1
Name: count, dtype: int64
avoid       ['essential_oils']  ['fragrance', 'essential_oils']  \
reactivity                                                        
False                        0                                0   
True                         1                                1   

avoid       ['fragrance']  []  
reactivity                     
False                   5  13  
True                    5   6  


,profile_id,kind,is_adult,concern,blemish_level,referral_flags,skin_type,reactivity,avoid,expected,expected_traits
0,P01,recommendation,True,acne_prone,2,[],oily,True,[],products,"fragrance-free first, not rich"
1,P02,recommendation,True,acne_prone,3,[],oily,True,[fragrance],products,"no parfum, fragrance-free first, not rich"
2,P03,recommendation,True,acne_prone,1,[],oily,False,[],products,not rich
3,P04,recommendation,True,acne_prone,2,[],oily,False,[fragrance],products,"no parfum, not rich"
4,P05,recommendation,True,acne_prone,3,[],combination,True,[],products,"fragrance-free first, not rich"
5,P06,recommendation,True,acne_prone,1,[],combination,True,[fragrance],products,"no parfum, fragrance-free first, not rich"
6,P07,recommendation,True,acne_prone,2,[],combination,False,[],products,not rich
7,P08,recommendation,True,acne_prone,3,[],combination,False,[fragrance],products,"no parfum, not rich"
8,P09,recommendation,True,acne_prone,1,[],normal,True,[],products,"fragrance-free first, not rich"
9,P10,recommendation,True,acne_prone,2,[],normal,True,[fragrance],products,"no parfum, fragrance-free first, not rich"


In [345]:
for profile in profiles:
    answers = {field: profile[field] for field in QUESTIONNAIRE}
    assert check_profile(answers) == [], profile["profile_id"]

for field in ["skin_type", "reactivity", "blemish_level"]:
    print(field, ":", sorted(profiles_table[field].astype(str).unique()))
print("avoid :", profiles_table["avoid"].astype(str).unique())

skin_type : ['combination', 'dry', 'normal', 'not_sure', 'oily']
reactivity : ['False', 'True']
blemish_level : ['1', '2', '3', '4', '5']
avoid : ['[]' "['fragrance']" "['essential_oils']"
 "['fragrance', 'essential_oils']"]


In [346]:
to_save = profiles_table.copy()
to_save["avoid"] = to_save["avoid"].apply("+".join)                     # ['fragrance', 'essential_oils'] -> 'fragrance+essential_oils'
to_save["referral_flags"] = to_save["referral_flags"].apply("+".join)   # [] -> '' (empty)
to_save.to_csv("../raw_data/swan_test_profiles_v1.csv", index=False)
print("Saved: swan_test_profiles_v1.csv")

Saved: swan_test_profiles_v1.csv


In [347]:
result_rows = []
for profile in profiles:
    result = recommend(profile, moist)
    if type(result) == str:
        result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": result})
    else:
        rank = 1
        for _, product in result.iterrows():           # one loop turn per recommended product (row)
            respects_avoid = True
            if "fragrance" in profile["avoid"] and product["has_parfum"]:
                respects_avoid = False
            if "essential_oils" in profile["avoid"] and product["has_essential_oil"]:
                respects_avoid = False
            if profile["skin_type"] == "dry":
                texture_ok = product["is_rich"]
            else:
                texture_ok = not product["is_rich"]
            result_rows.append({"profile_id": profile["profile_id"], "expected": profile["expected"], "outcome": "products",
                                "rank": rank, "code": product["code"], "product_name": product["product_name"],
                                "score": product["score"], "R2_respects_avoid": respects_avoid,
                                "expected_traits": profile["expected_traits"], "texture_ok": texture_ok,
                                "R4_face_moisturizer": "", "R5_texture_fits": "", "comment": ""})
            rank = rank + 1

results = pd.DataFrame(result_rows)
results.head(10)

,profile_id,expected,outcome,rank,code,product_name,score,R2_respects_avoid,expected_traits,texture_ok,R4_face_moisturizer,R5_texture_fits,comment
0,P01,products,products,1.0,5017634247843,Anti-boutons,1.0,True,"fragrance-free first, not rich",True,,,
1,P01,products,products,2.0,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,1.0,True,"fragrance-free first, not rich",True,,,
2,P01,products,products,3.0,3401360106994,Sébium sensitive,1.0,True,"fragrance-free first, not rich",True,,,
3,P02,products,products,1.0,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,1.0,True,"no parfum, fragrance-free first, not rich",True,,,
4,P02,products,products,2.0,3701129804445,Créaline Défensive,1.0,True,"no parfum, fragrance-free first, not rich",True,,,
5,P02,products,products,3.0,3605970360757,Ultra Facial Cream,1.0,True,"no parfum, fragrance-free first, not rich",True,,,
6,P03,products,products,1.0,5017634247843,Anti-boutons,2.0,True,not rich,True,,,
7,P03,products,products,2.0,3401360106994,Sébium sensitive,2.0,True,not rich,True,,,
8,P03,products,products,3.0,3380813064107,Gel Revitalisant,2.0,True,not rich,True,,,
9,P04,products,products,1.0,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,1.0,True,"no parfum, not rich",True,,,


In [348]:
outcome_per_profile = results.groupby("profile_id")[["expected", "outcome"]].first()
print("R1 expected outcome     :", (outcome_per_profile["expected"] == outcome_per_profile["outcome"]).sum(), "/", len(outcome_per_profile))

products = results[results["outcome"] == "products"]
print("R2 respects 'avoid'     :", products["R2_respects_avoid"].sum(), "/", len(products))
print("R3 no duplicate in a top:", (products.groupby("profile_id")["code"].nunique() == NB_ROUTINE_PRODUCT).sum(), "/", products["profile_id"].nunique())
print("Texture fits (automatic):", products["texture_ok"].sum(), "/", len(products), "  (rule: rich for dry skin, not rich otherwise)")

print()
print("Profiles with a wrong outcome:")
outcome_per_profile[outcome_per_profile["expected"] != outcome_per_profile["outcome"]]

R1 expected outcome     : 31 / 31
R2 respects 'avoid'     : 69 / 69
R3 no duplicate in a top: 23 / 23
Texture fits (automatic): 68 / 69   (rule: rich for dry skin, not rich otherwise)

Profiles with a wrong outcome:


,expected,outcome
profile_id,,


In [349]:
results.to_csv("../raw_data/swan_test_results_v1.csv", index=False)
print("Saved: swan_test_results_v1.csv —", len(products), "products to judge")

Saved: swan_test_results_v1.csv — 69 products to judge


## Extra checks

In [350]:
tiny = moist[moist["in_scope"]].head(2)       # a catalogue with only 2 products
base_profile = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                "skin_type": "oily", "reactivity": False, "avoid": []}
assert recommend(base_profile, tiny) == "not enough products"

no_avoid = {k: v for k, v in base_profile.items() if k != "avoid"}   # Q7 not answered
assert type(recommend(no_avoid, moist)) != str                       # gets products, no crash
print("Extra checks OK")

Extra checks OK


## Tests on fictional products

6 invented products with known answers: we check the logic of `recommend()` without depending on the real data. These products are never recommended.

In [351]:
fake = pd.DataFrame({
    "code":          ["F1", "F2", "F3", "F4", "F5", "F6"],
    "product_name":  ["Light gel", "Light fluid", "Rich cream", "Rich balm", "Parfum gel", "Oil cream"],
    "name_brand":    ["light gel", "light fluid", "rich cream", "rich balm", "parfum gel", "oil cream"],
    "in_scope":      [True, True, True, True, True, True],
    "has_parfum":    [False, False, False, False, True, False],
    "has_fragrance_allergen": [False, False, False, False, True, False],
    "has_essential_oil": [False, False, False, False, False, True],
    "has_acne_active": [False, False, False, False, False, False],
    "is_anti_ageing": [False, False, False, False, False, False],
    "is_light":      [True, True, False, False, True, False],
    "is_rich":       [False, False, True, True, False, True],
    "coverage":      [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
})
base_profile = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
                "skin_type": "oily", "reactivity": False, "avoid": []}

def change(field, value):
    """A copy of base_profile with one answer changed."""
    profile = base_profile.copy()
    profile[field] = value
    return profile

# Routing
assert recommend(change("is_adult", False), fake) == "referral"
assert recommend(change("concern", "other"), fake) == "out of scope"
assert recommend(change("blemish_level", 4), fake) == "referral"

# Avoid answers are respected
result = recommend(change("avoid", ["fragrance"]), fake)
assert "F5" not in list(result["code"])
result = recommend(change("avoid", ["essential_oils"]), fake)
assert "F6" not in list(result["code"])

# Texture: light first for oily skin, rich first for dry skin
assert recommend(base_profile, fake).iloc[0]["is_light"] == True
assert recommend(change("skin_type", "dry"), fake).iloc[0]["is_rich"] == True

# The same product (same name + brand) never twice
twins = fake.copy()
twins.loc[1, "name_brand"] = "light gel"
result = recommend(base_profile, twins)
assert result["name_brand"].nunique() == len(result)

print("Tests on fictional OK")

Tests on fictional OK


In [352]:
print("1 under 18      :", recommend(change("is_adult", False), fake))
print("2 other concern :", recommend(change("concern", "other"), fake))
print("3 blemishes 4   :", recommend(change("blemish_level", 4), fake))

result = recommend(change("avoid", ["fragrance"]), fake)
print("4 avoid parfum  :", result if type(result) == str else list(result["code"]))
result = recommend(change("avoid", ["essential_oils"]), fake)
print("5 avoid oils    :", result if type(result) == str else list(result["code"]))

result = recommend(base_profile, fake)
print("6 oily          :", result if type(result) == str else list(result["code"]))
result = recommend(change("skin_type", "dry"), fake)
print("7 dry           :", result if type(result) == str else list(result["code"]))

1 under 18      : referral
2 other concern : out of scope
3 blemishes 4   : referral
4 avoid parfum  : ['F1', 'F2', 'F3']
5 avoid oils    : ['F1', 'F2', 'F5']
6 oily          : ['F1', 'F2', 'F5']
7 dry           : ['F3', 'F4', 'F6']


## Does each question change the result?

Start from one profile and change **one answer at a time**.

In [353]:
base = {"is_adult": True, "concern": "acne_prone", "blemish_level": 2, "referral_flags": [],
        "skin_type": "oily", "reactivity": False, "avoid": []}

for question in ["skin_type", "reactivity", "avoid"]:
    print("=====", question)
    for answer in QUESTIONNAIRE[question]:
        profile = base.copy()
        profile[question] = answer
        result = recommend(profile, moist)
        if type(result) == str:
            print(answer, "→", result)
        else:
            print(answer, "→", list(result["product_name"]))

===== skin_type
oily → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
combination → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
normal → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
dry → ['Crème riche apaisante pro-tolérance', 'Soin protecteur regénérant Thé vert + Vigne rouge régénérante', 'Anti-boutons']
not_sure → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
===== reactivity
True → ['Anti-boutons', 'ISNTREE Hyaluronic Acid Moist Cream', 'Sébium sensitive']
False → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
===== avoid
[] → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
['fragrance'] → ['ISNTREE Hyaluronic Acid Moist Cream', 'Créaline Défensive', 'Ultra Facial Cream']
['essential_oils'] → ['Anti-boutons', 'Sébium sensitive', 'Gel Revitalisant']
['fragrance', 'essential_oils'] → ['ISNTREE Hyaluronic Acid Moist Cream', 'Créaline Défensive', 'Ultra Facial Cream']


In [354]:
tops = products.groupby("profile_id")["product_name"].apply(" + ".join)
print("Different top 3:", tops.nunique(), "out of", len(tops), "profiles with products")
print("Different products shown:", products["code"].nunique(), "out of", moist["in_scope"].sum(), "in scope")
print()
print("Most shown products:")
products["product_name"].value_counts().head(5)

Different top 3: 5 out of 23 profiles with products
Different products shown: 10 out of 99 in scope

Most shown products:


product_name
ISNTREE Hyaluronic Acid Moist Cream    15
Anti-boutons                           11
Sébium sensitive                       10
Créaline Défensive                      9
Ultra Facial Cream                      9
Name: count, dtype: int64